# GenoPhenoIR — Exploration Notebook

This notebook demonstrates the full GenoPhenoIR pipeline on Arabidopsis thaliana
chromosome 1, exploring whether inverted repeat (IR) patterns correlate with
phenotypic variation across accessions.

**Phases:**
1. IR Profiling — detect IRs on the reference and build per-accession fingerprints
2. Phenotype Integration — merge IR data with flowering time, leaf number, growth rate
3. Pattern Discovery — UMAP clustering, RF/GBR prediction, SHAP explainability
4. Visual Encoding — image-based representation + CNN autoencoder

In [ ]:
import sys
import os

# Ensure the project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
%matplotlib inline

from genophenoir.config_loader import load_config, resolve_path
cfg = load_config('config.yaml')
print(f'Project root: {cfg.project_root}')
print(f'Chromosomes: {cfg.params.chromosomes}')
print(f'Max accessions: {cfg.params.max_accessions}')

## Phase 1 — IR Profiling

Scan the TAIR10 reference genome for inverted repeats and build a per-accession
binary fingerprint matrix from VCF variants.

In [ ]:
from genophenoir.ir_profiler import run_phase1

ir_list, fingerprint_df = run_phase1(cfg)

print(f'\nInverted repeats found: {len(ir_list)}')
print(f'Fingerprint matrix shape: {fingerprint_df.shape}')
print(f'Disruption rate: {(fingerprint_df == 0).mean().mean():.2%}')

In [ ]:
# Visualise IR distribution along chromosome 1
ir_df = pd.DataFrame([{
    'start': ir.start, 'end': ir.end,
    'stem_length': ir.stem_length, 'spacer_length': ir.spacer_length,
    'midpoint': (ir.start + ir.end) // 2,
    'total_length': ir.end - ir.start,
} for ir in ir_list])

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# IR density along the chromosome
axes[0, 0].hist(ir_df['midpoint'] / 1e6, bins=60, color='steelblue', edgecolor='white')
axes[0, 0].set_xlabel('Genomic position (Mb)')
axes[0, 0].set_ylabel('IR count')
axes[0, 0].set_title('IR density along Chr1')

# Stem length distribution
axes[0, 1].hist(ir_df['stem_length'], bins=30, color='coral', edgecolor='white')
axes[0, 1].set_xlabel('Stem length (bp)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Stem length distribution')

# Spacer length distribution
axes[1, 0].hist(ir_df['spacer_length'], bins=30, color='seagreen', edgecolor='white')
axes[1, 0].set_xlabel('Spacer length (bp)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Spacer length distribution')

# Fingerprint disruption rate per IR
disruption_rate = (fingerprint_df == 0).mean(axis=0)
axes[1, 1].hist(disruption_rate, bins=30, color='mediumpurple', edgecolor='white')
axes[1, 1].set_xlabel('Disruption rate across accessions')
axes[1, 1].set_ylabel('Number of IRs')
axes[1, 1].set_title('IR disruption rate distribution')

plt.tight_layout()
plt.savefig('output/phase1/ir_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## Phase 2 — Phenotype Integration

Load phenotype data and merge with the IR fingerprint matrix.

In [ ]:
from genophenoir.phenotype_loader import run_phase2

merged_df = run_phase2(cfg)

print(f'\nMerged dataset: {merged_df.shape[0]} accessions, {merged_df.shape[1]} columns')

# Identify trait columns
trait_cols = [c for c in cfg.params.target_traits if c in merged_df.columns]
print(f'Trait columns: {trait_cols}')
print(f'\nTrait summary statistics:')
merged_df[trait_cols].describe()

In [ ]:
# Visualise phenotype distributions
fig, axes = plt.subplots(1, len(trait_cols), figsize=(5 * len(trait_cols), 4))
if len(trait_cols) == 1:
    axes = [axes]

colors = ['steelblue', 'coral', 'seagreen']
for ax, trait, color in zip(axes, trait_cols, colors):
    vals = merged_df[trait].dropna()
    ax.hist(vals, bins=30, color=color, edgecolor='white', alpha=0.8)
    ax.set_xlabel(trait.replace('_', ' ').title())
    ax.set_ylabel('Count')
    ax.axvline(vals.mean(), color='red', linestyle='--', alpha=0.7, label=f'mean={vals.mean():.1f}')
    ax.legend()

plt.suptitle('Phenotype Distributions', fontsize=14)
plt.tight_layout()
plt.savefig('output/phase2/phenotype_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Phase 3 — Pattern Discovery

### 3a. Clustering (UMAP + HDBSCAN)

Cluster accessions by IR fingerprint and check if clusters correlate with phenotype.

In [ ]:
from genophenoir.pattern_analysis import _split_features_targets, run_clustering

X, y = _split_features_targets(merged_df, cfg.params.target_traits)
print(f'Feature matrix: {X.shape}')
print(f'Target matrix: {y.shape}')

out_dir = resolve_path(cfg, cfg.paths.phase3_output)
out_dir.mkdir(parents=True, exist_ok=True)

umap_df = run_clustering(X, y, cfg, out_dir)

print(f'\nCluster counts:')
print(umap_df['cluster'].value_counts().sort_index())

In [ ]:
# Interactive-style cluster visualisation with phenotype overlay
if len(trait_cols) > 0:
    fig, axes = plt.subplots(1, len(trait_cols) + 1, figsize=(6 * (len(trait_cols) + 1), 5))
    if len(trait_cols) + 1 == 1:
        axes = [axes]

    # Cluster plot
    sc = axes[0].scatter(
        umap_df['UMAP1'], umap_df['UMAP2'],
        c=umap_df['cluster'], cmap='tab20', s=12, alpha=0.7
    )
    axes[0].set_title('HDBSCAN Clusters')
    axes[0].set_xlabel('UMAP 1')
    axes[0].set_ylabel('UMAP 2')
    plt.colorbar(sc, ax=axes[0], label='Cluster')

    # Phenotype overlays
    for i, trait in enumerate(trait_cols):
        vals = y[trait].reindex(umap_df.index)
        mask = vals.notna()
        sc = axes[i + 1].scatter(
            umap_df.loc[mask, 'UMAP1'], umap_df.loc[mask, 'UMAP2'],
            c=vals[mask], cmap='viridis', s=12, alpha=0.7
        )
        axes[i + 1].set_title(f'Coloured by {trait}')
        axes[i + 1].set_xlabel('UMAP 1')
        axes[i + 1].set_ylabel('UMAP 2')
        plt.colorbar(sc, ax=axes[i + 1], label=trait)

    plt.tight_layout()
    plt.show()

### 3b. Predictive Modelling

Train Random Forest and Gradient Boosting regressors to predict phenotype from
IR fingerprints, with shuffled-feature baseline comparison.

In [ ]:
from genophenoir.pattern_analysis import run_prediction

pred_results = run_prediction(X, y, cfg, out_dir)

# Display results table
summary_rows = []
for trait, models in pred_results.items():
    for model_name, metrics in models.items():
        summary_rows.append({
            'Trait': trait,
            'Model': model_name,
            'R^2': f"{metrics['r2']:.4f}",
            'R^2 CV': f"{metrics['r2_cv_mean']:.4f} +/- {metrics['r2_cv_std']:.4f}",
            'Pearson r': f"{metrics['pearson_r']:.4f}",
            'p-value': f"{metrics['pearson_pval']:.2e}",
            'Baseline R^2': f"{metrics['baseline_r2_mean']:.4f} +/- {metrics['baseline_r2_std']:.4f}",
        })

pd.DataFrame(summary_rows)

### 3c. SHAP Explainability

Identify which IR regions are most predictive of each phenotype.

In [ ]:
from genophenoir.pattern_analysis import run_shap_analysis

gff3_path = resolve_path(cfg, cfg.paths.gff3_file)
shap_results = run_shap_analysis(X, y, cfg, out_dir, gff3_path)

# Show top features for each trait
for trait, top_df in shap_results.items():
    print(f'\n=== Top SHAP features for {trait} ===')
    display(top_df.head(10))

## Phase 4 — Visual Encoding Explorer

Create 2D image encodings of per-accession IR profiles, train a CNN autoencoder,
and compare the learned latent space with the tabular clustering from Phase 3.

In [ ]:
from genophenoir.image_encoder import run_phase4

phase4_results = run_phase4(cfg)

latent_vectors = phase4_results['latent_vectors']
img_umap = phase4_results['img_umap']

print(f'Latent vectors shape: {latent_vectors.shape}')
print(f'Image UMAP shape: {img_umap.shape}')

In [ ]:
# Show sample accession images
from PIL import Image

img_dir = resolve_path(cfg, cfg.paths.phase4_output) / 'images'
sample_images = sorted(img_dir.glob('*.png'))[:6]

if sample_images:
    fig, axes = plt.subplots(2, 3, figsize=(15, 6))
    for ax, img_path in zip(axes.flat, sample_images):
        img = Image.open(img_path)
        ax.imshow(np.array(img), cmap='gray', aspect='auto')
        ax.set_title(img_path.stem, fontsize=9)
        ax.axis('off')
    plt.suptitle('Sample Accession Image Encodings', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No images found. Run Phase 4 first.')

In [ ]:
# Compare tabular vs image-based UMAP
tab_umap_path = resolve_path(cfg, cfg.paths.phase3_output) / 'umap_clusters.csv'
if tab_umap_path.exists():
    tab_umap = pd.read_csv(tab_umap_path, index_col=0)
    common = img_umap.index.intersection(tab_umap.index)

    if len(common) > 10:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        axes[0].scatter(
            tab_umap.loc[common, 'UMAP1'],
            tab_umap.loc[common, 'UMAP2'],
            c=tab_umap.loc[common, 'cluster'], cmap='tab20', s=12, alpha=0.7
        )
        axes[0].set_title('Tabular IR Fingerprint (Phase 3)')
        axes[0].set_xlabel('UMAP 1')
        axes[0].set_ylabel('UMAP 2')

        axes[1].scatter(
            img_umap.loc[common, 'UMAP1_img'],
            img_umap.loc[common, 'UMAP2_img'],
            c=tab_umap.loc[common, 'cluster'], cmap='tab20', s=12, alpha=0.7
        )
        axes[1].set_title('Image Autoencoder Latent (Phase 4)')
        axes[1].set_xlabel('UMAP 1')
        axes[1].set_ylabel('UMAP 2')

        plt.suptitle('Tabular vs Image-based Clustering (coloured by Phase 3 cluster)', fontsize=13)
        plt.tight_layout()
        plt.show()
    else:
        print(f'Only {len(common)} common accessions; need > 10 for comparison.')
else:
    print('Phase 3 UMAP output not found. Run Phase 3 first.')

## Summary

This notebook explored whether inverted repeat variation patterns across
Arabidopsis accessions correlate with phenotypic traits. Key outputs:

- **Phase 1**: IR detection and per-accession fingerprinting
- **Phase 2**: Merged genotype-phenotype dataset
- **Phase 3**: Clustering, prediction accuracy, and SHAP feature importance
- **Phase 4**: Image-based encoding comparison

Check the `output/` directory for all generated plots and data files.